In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [2]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [3]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [5]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [6]:
square_root.invoke({"x": 467})

21.61018278497431

In [7]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

## 测试

In [8]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-chat",
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

D:\CODE\jc-course\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [9]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


好的，我来查询杭州的天气，并包含未来几天的预报信息。Current weather in 杭州: 22 degrees C
Next 5 days: Sunny杭州接下来的天气情况如下：

🌤 **当前天气**：22°C

📅 **未来5天预报**：以**晴天**为主，天气不错！

看起来杭州接下来几天都是晴朗的好天气，温度也比较舒适。如果你想知道更具体的每日气温（比如最高温和最低温），可以告诉我，我再帮你详细查询！😊

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少?
None
================================== Ai Message ==================================

我来计算这两个数的平方根。
Tool Calls:
  square_root (call_00_Yx1SqGID82fpzULoujC56378)
 Call ID: call_00_Yx1SqGID82fpzULoujC56378
  Args:
    x: 467
  square_root (call_01_UJrhJRaCisyVZR4xuIgh8100)
 Call ID: call_01_UJrhJRaCisyVZR4xuIgh8100
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

计算得出：

- **467的平方根** ≈ **21.610**（精确到小数点后三位）
- **529的平方根** = **23**（整数的平方，因为 23 × 23 = 529）
None


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [11]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [12]:
search_tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.sina.cn/news/detail/5251386149438121.html',
   'title': '蒸蚌梗稀缺性引关注_新浪新闻',
   'content': '原来这一声单纯的蒸蚌是这么稀缺的东西。 这才是这个梗能爆出来的真正原因吧。 原来我都没往这边想，这种视频看多了差点脱口而出给别人喊蒸蚌。',
   'score': 0.76173395,
   'raw_content': None},
  {'url': 'https://www.huxiu.com/article/4820519.html',
   'title': '一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？ - 虎嗅',
   'content': '一只名叫“超级无敌大开门”的三花猫，因主人教它辨认“萝卜”和“纸巾”的视频走红。其“连蒙带猜”的笨拙表现引发了网友大规模的二创模仿和讨论，最终大家发现，这只“文化课”不佳的小猫，在“体育课”和人情世故上却是天才，其爆火现象折射出人们的情感共鸣与自嘲心态。 ## 梗的起源：从“纸巾”到“萝卜”的猫猫课堂 - 视频主角是猫咪“大开门”，主人通过零食奖励和“蒸蚌！”的鼓励教它辨认物品。 - 尽管教学氛围积极，但开门学习进度缓慢，常常指错答案（如把鼠标当成纸巾），甚至试图“逃课”。 ## 病毒式传播：为何“笨猫”能击中大众心巴？ - 开门犹豫试探、试图“萌”混过关的神情，让网友联想到学生时代面对考试的自己。 - 主人洗脑的“蒸蚌！”称赞与开门实际“蒙题”的反差，形成了极具传播力的喜剧效果。 ## 二创狂欢：从宠物模仿到“无猫”赛道 - 由于道具简单、易于模仿，大量网友和宠物参与了二创，内容从同类宠物扩展到人类乃至2D动画。 - 梗的应用范围被拓宽，甚至被各地文旅官号用来宣传特色美食（如河南烩面、江西瓦罐汤）。 ## 形象反转：“体育生”的才华与大众的情感投射 - 网友通过给自家猫“考试”发现，开门能安静配合拍摄已超越了99%的猫咪。 - 后续视频显示开门能完美完成“跳圈”等指令，网友顿悟它是“体育生”

In [13]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [14]:
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

蒸蚌是什么梗？
================================== Ai Message ==================================

让我搜索一下这个网络流行语的来源和含义。
Tool Calls:
  tavily_search (call_00_fP5ZMxz6DsDBzMhTFXAl0769)
 Call ID: call_00_fP5ZMxz6DsDBzMhTFXAl0769
  Args:
    query: 蒸蚌 梗 是什么意思 网络用语
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 梗 是什么意思 网络用语", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.huxiu.com/article/4820519.html", "title": "一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？ - 虎嗅", "content": "一只名叫“超级无敌大开门”的三花猫，因主人教它辨认“萝卜”和“纸巾”的视频走红。其“连蒙带猜”的笨拙表现引发了网友大规模的二创模仿和讨论，最终大家发现，这只“文化课”不佳的小猫，在“体育课”和人情世故上却是天才，其爆火现象折射出人们的情感共鸣与自嘲心态。 ## 梗的起源：从“纸巾”到“萝卜”的猫猫课堂 - 视频主角是猫咪“大开门”，主人通过零食奖励和“蒸蚌！”的鼓励教它辨认物品。 - 尽管教学氛围积极，但开门学习进度缓慢，常常指错答案（如把鼠标当成纸巾），甚至试图“逃课”。 ## 病毒式传播：为何“笨猫”能击中大众心巴？ - 开门犹豫试探、试图“萌”混过关的神情，让网友联想到学生时代面对考试的自己。 - 主人洗脑的“蒸蚌！”称赞与开门实际“蒙题”的反差，形成了极具传播力的喜剧效果。 ## 二创狂欢

## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [15]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [16]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [17]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [18]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='## 「蒸蚌」是什么梗？\n\n**「蒸蚌」是「真棒」的谐音空耳（听错），源自近期火遍全网的抖音萌宠博主「超级无敌大开门」的系列视频。**\n\n### 梗的起源\n\n博主「超级无敌大开门」养了一只叫**「大开门」**的三花猫，她经常教猫咪辨认各种物品（如胡萝卜🥕、纸巾🧻、米老鼠🐭等）。每当猫咪指对答案时，主人就会用非常夸张、高亢的语调喊一声**「真棒！」**来鼓励它。\n\n但由于主人发音比较用力且语调魔性，听起来非常像**「蒸蚌！」**，加上猫咪常常一脸懵、连蒙带猜地乱指一通（有时指鼠标当纸巾、有时干脆想逃课），这种**「笨猫答错题 + 主人依然夸蒸蚌」**的反差萌，产生了极其洗脑的喜剧效果。\n\n### 走红与二创\n\n这个梗迅速在抖音、B站、小红书等平台病毒式传播：\n\n- 大量网友模仿拍摄自家宠物做「萝卜 or 纸巾」的辨认挑战\n- 后来发展到人类、二次元角色、甚至文旅官号也加入二创（如用河南烩面、江西瓦罐汤替代萝卜和纸巾）\n- 猫咪「大开门」也被网友称为「萝卜纸巾猫」「体育生」（因为它文化课不行但运动能力超强）\n\n### 使用方法\n\n大家在评论区或聊天中发「蒸蚌」就是表示**「真棒」「太牛了」**的意思，带有一种幽默、调侃、自嘲的语气。\n\n### 总结\n\n> 蒸蚌 = 真棒（谐音梗），源自一只笨猫和它洗脑的铲屎官，表达夸奖但自带搞笑和反差感 🐱✨' reference=[Reference(title='"我蒸蚌"是什么意思？ - HiNative', url='https://zh.hinative.com/questions/15904858'), Reference(title='现在互联网上最火的猫，从"耄耋"变成了"萝卜纸巾猫" - 搜狐', url='https://www.sohu.com/a/972125782_258858'), Reference(title='一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？ - 虎嗅', url='https://www.huxiu.com/article/4820519.html'), Reference(title='【梗百科】萝卜纸巾猫是啥梗？真棒！- Bilibili', url='https://www.bilibili.com/video